# ADS-B contextual_physics_v2 — Colab CUDA koşusu

Bu notebook dondurulmuş veri, scaler, 8 epoch, batch=512, model ve loss sözleşmesini değiştirmez. Yerel koşudaki iki operasyonel sorunu düzeltir: model/batch'leri CUDA'ya taşır ve batch düzeyinde ilerleme + kesinti checkpoint'i üretir.

Colab koparsa 1–8. hücreleri yeniden çalıştırın. Eğitim hücresi Google Drive'daki `colab_resume_checkpoint.pt` dosyasından aynı epoch/parça/batch konumunda devam eder.

## 1. Drive'ı bağla ve yolları ayarla

Yerelde üretilen `artifacts/adsb/colab/contextual_v2_transfer/` klasöründeki **bütün dosyaları** Drive'da tek klasöre yükleyin.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BUNDLE_DIR = Path('/content/drive/MyDrive/adsb_contextual_v2_transfer')  # gerekirse değiştir
WORK_ROOT = Path('/content/adsb_contextual_v2')
RUN_DIR = Path('/content/drive/MyDrive/adsb_runs/20260725_contextual_physics_v2_colab_cuda_v1')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
print('BUNDLE_DIR:', BUNDLE_DIR)
print('WORK_ROOT :', WORK_ROOT)
print('RUN_DIR   :', RUN_DIR)

## 2. GPU'yu fail-loudly doğrula

`Runtime > Change runtime type > T4 GPU` seçili değilse bu hücre durur. CPU fallback kasıtlı olarak yoktur.

In [ ]:
import torch, platform
print('Python:', platform.python_version())
print('Torch :', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime seçili değil; CPU ile devam edilmiyor.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

## 3. Transfer shard'larını doğrula ve yerel Colab diskine aç

ZIP shard'ları Parquet'i yeniden sıkıştırmaz. Her runtime yeniden başladığında Drive'dan `/content` diskine açılır; tamamlanmış shard marker'ları aynı oturum içinde tekrar açmayı engeller.

In [ ]:
import hashlib, json, shutil, time, zipfile

def sha256_file(path, block=8*1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(block):
            h.update(chunk)
    return h.hexdigest()

index_path = BUNDLE_DIR / 'transfer_index.json'
index = json.loads(index_path.read_text(encoding='utf-8'))
if not index.get('archives_built'):
    raise RuntimeError('Bundle archive içermiyor; yerelde --build-archives ile yeniden üretin.')
marker_dir = WORK_ROOT / '.extracted_shards'
marker_dir.mkdir(parents=True, exist_ok=True)
for number, record in enumerate(index['archives'], 1):
    archive = BUNDLE_DIR / record['path']
    marker = marker_dir / (archive.name + '.ok')
    if marker.exists() and marker.read_text().strip() == record['sha256']:
        print(f'[{number}/{len(index["archives"])}] hazır: {archive.name}')
        continue
    started = time.perf_counter()
    observed = sha256_file(archive)
    if observed != record['sha256']:
        raise RuntimeError(f'Archive SHA-256 uyuşmazlığı: {archive}')
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(WORK_ROOT)
    marker.write_text(observed)
    print(f'[{number}/{len(index["archives"])}] açıldı: {archive.name} ({time.perf_counter()-started:.1f}s)')
shutil.copy2(BUNDLE_DIR / 'bundle_manifest.json', WORK_ROOT / 'bundle_manifest.json')
print('Extraction complete:', WORK_ROOT)

## 4. Minimal bağımlılıkları kur

Colab'ın CUDA uyumlu Torch kurulumu korunur; Torch yeniden kurulmaz.

In [ ]:
%pip install -q pandas pyarrow scipy tqdm
import pandas, pyarrow, scipy
print('pandas', pandas.__version__, 'pyarrow', pyarrow.__version__, 'scipy', scipy.__version__)

## 5. Çıkarılmış 784 Parquet'i tam SHA-256 ile doğrula

Bu hücre kontrat kapısıdır. Herhangi bir byte/hash uyuşmazlığında eğitim başlamaz.

In [ ]:
import os, subprocess, sys
env = os.environ.copy()
env['PYTHONPATH'] = str(WORK_ROOT)
cmd = [sys.executable, str(WORK_ROOT/'scripts/adsb_contextual_v2_colab_runner.py'),
       'verify', '--repo-root', str(WORK_ROOT),
       '--bundle-manifest', str(WORK_ROOT/'bundle_manifest.json'),
       '--verification', 'full']
subprocess.run(cmd, check=True, env=env)

## 6. Veri ölçeğini göster

Beklenen: 784 parça, 275,468,643 fiziksel satır, yaklaşık 17.674 GiB.

In [ ]:
manifest = json.loads((WORK_ROOT/'bundle_manifest.json').read_text(encoding='utf-8'))
print(json.dumps(manifest['totals'], indent=2))
assert manifest['totals']['parts'] == 784
assert manifest['totals']['parquet_rows'] == 275_468_643

## 7. İki parçada gerçek GPU benchmark

Bu hücre feature/window süresini GPU backprop süresinden ayırır. Sonuç görülse bile frozen batch/epoch/model değerleri değiştirilmez.

In [ ]:
cmd = [sys.executable, str(WORK_ROOT/'scripts/adsb_contextual_v2_colab_runner.py'),
       'benchmark', '--repo-root', str(WORK_ROOT),
       '--bundle-manifest', str(WORK_ROOT/'bundle_manifest.json'),
       '--verification', 'sizes', '--device', 'cuda', '--parts', '2']
subprocess.run(cmd, check=True, env=env)

## 8. Eğitimi başlat veya checkpoint'ten devam et

Her 200 batch'te canlı loss/VRAM/konum yazılır. Her 1.000 batch'te ve her Parquet sonunda Drive'a atomik resume checkpoint'i yazılır. Bu hücre uzun çalışır; Colab bağlantısı koparsa üst hücreleri ve bu hücreyi yeniden çalıştırın.

In [ ]:
cmd = [sys.executable, str(WORK_ROOT/'scripts/adsb_contextual_v2_colab_runner.py'),
       'train', '--repo-root', str(WORK_ROOT),
       '--bundle-manifest', str(WORK_ROOT/'bundle_manifest.json'),
       '--run-dir', str(RUN_DIR), '--verification', 'sizes', '--device', 'cuda',
       '--progress-every-batches', '200', '--checkpoint-every-batches', '1000']
subprocess.run(cmd, check=True, env=env)

## 9. Zorunlu magnitude kapısını göster

`true` ise **DUR** ve Faz D'ye geçme. `false` ise run klasörünü yerel repoya indir; Faz D–G aynı frozen sözleşmeyle orada devam eder.

In [ ]:
report_path = RUN_DIR / 'training_report.json'
if not report_path.exists():
    checkpoint = RUN_DIR / 'colab_resume_checkpoint.pt'
    print('Henüz tamamlanmadı. Resume checkpoint:', checkpoint, checkpoint.exists())
else:
    report = json.loads(report_path.read_text(encoding='utf-8'))
    diag = report['natural_calibration_diagnostic']
    print(json.dumps(diag, indent=2))
    flagged = diag['magnitude_domination_flagged_at_0_8']
    print('MAGNITUDE GATE:', 'STOP / TRUE' if flagged else 'PASS / FALSE')
    if flagged:
        raise RuntimeError("Magnitude domination true: Faz D'ye geçmeyin.")